# Reviewer Forecast Diagnostics

This notebook generates the compact forecasting and scenario-generation results needed for the reviewer comment:

> The manuscript uses quantile-based XGBoost forecasting models to generate uncertainty scenarios, but the details regarding scenario generation quality, calibration accuracy, and statistical properties are limited.

The notebook reports only the requested results:

1. **Point forecasting metrics** for temperature and net load (`load - PV`): `MAPE`, `MAE`, `RMSE`, `nMAE`, and `nRMSE`.
2. **Quantile/scenario diagnostics**: `PICP`, scenario mean error, scenario standard deviation, 5th quantile, and 95th quantile.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

cwd = Path.cwd().resolve()
if (cwd / "UATMPC").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == "UATMPC":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path("/Users/liqi/Desktop/UATMPC codes")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from UATMPC.config import MPCConfig, Paths
from UATMPC.data import MONTH_DAYS, build_case_data, build_data_y_window, load_all_data
from UATMPC.forecasting import QuantileForecaster, enforce_non_crossing
from UATMPC.scenarios import ScenarioGenerator, dict_to_array

pd.set_option("display.max_columns", 80)
PROJECT_ROOT


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/liqi/Desktop/UATMPC codes')

## 1. Load Data and Forecasting Models


In [2]:
config = MPCConfig()
paths = Paths.from_root(PROJECT_ROOT)

data = load_all_data(paths, config)

temp_model = QuantileForecaster(paths.temp_model_dir).load()
pv_model = QuantileForecaster(paths.pv_model_dir).load()
load_model = QuantileForecaster(paths.load_model_dir).load()
scenario_generator = ScenarioGenerator(temp_model, pv_model, load_model, config)

quantiles = tuple(config.quantiles)
horizon = config.control_horizon
median_q = 0.5
median_idx = quantiles.index(median_q)

print(f"Temperature model: {paths.temp_model_dir}")
print(f"PV model: {paths.pv_model_dir}")
print(f"Load model: {paths.load_model_dir}")
print(f"Quantiles: {quantiles}")
print(f"Forecast horizon: {horizon} steps")


Temperature model: /Users/liqi/Desktop/UATMPC codes/15temp_models
PV model: /Users/liqi/Desktop/UATMPC codes/15pv_models
Load model: /Users/liqi/Desktop/UATMPC codes/15load_models
Quantiles: (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
Forecast horizon: 8 steps


## 2. Evaluation Settings

`MAX_DATES = 14` is useful for a quick check. For final manuscript results, set `MAX_DATES = None` to evaluate the full holdout period.


In [3]:
all_dates = [
    (month, day)
    for month in range(7, 8)
    for day in range(2, int(MONTH_DAYS[month - 1]) + 1)
]

EVAL_DATES = all_dates[:]

MAX_DATES = 50 # Change to None for final full holdout results.
STEP_STRIDE = 1

if MAX_DATES is not None:
    EVAL_DATES = EVAL_DATES[-MAX_DATES:]

print(f"Evaluation dates: {len(EVAL_DATES)}")
print(EVAL_DATES[:5], "...", EVAL_DATES[-5:])


Evaluation dates: 30
[(7, 2), (7, 3), (7, 4), (7, 5), (7, 6)] ... [(7, 27), (7, 28), (7, 29), (7, 30), (7, 31)]


## 3. Point Forecast Metrics

This section evaluates the point forecasts used by the MPC: temperature and net load (`load - PV`). For consistency with the uncertainty representation used by MPC, the point forecast is the mean of the generated scenarios at each horizon (not the median quantile forecast).

Reported metrics:

- `MAPE_% = mean(|prediction - truth| / |truth|) * 100`
- `MAE = mean(|prediction - truth|)`
- `RMSE = sqrt(mean((prediction - truth)^2))`
- `nMAE_% = MAE / (max(truth) - min(truth)) * 100`
- `nRMSE_% = RMSE / (max(truth) - min(truth)) * 100`


In [4]:
def q_col(prefix, q):
    return f"{prefix}_q{int(round(q * 100)):02d}"


def forecast_step_quantiles(case_data, t):
    curr_x_temp = case_data["x_temp"][t : t + 1, :]
    curr_x_pv = case_data["x_pv"][t : t + 1, :]
    curr_x_load = case_data["x_load"][t : t + 1, :]

    temp_preds = enforce_non_crossing(temp_model.predict(curr_x_temp, quantiles), quantiles)
    temp_q = dict_to_array(temp_preds, horizon, quantiles)

    # PV and load models use the temperature forecast as an additional input.
    temp_point = temp_q[:, median_idx].reshape(1, -1)

    pv_preds = enforce_non_crossing(
        pv_model.predict(np.hstack([curr_x_pv, temp_point]), quantiles),
        quantiles,
    )
    load_preds = enforce_non_crossing(
        load_model.predict(np.hstack([curr_x_load, temp_point]), quantiles),
        quantiles,
    )

    pv_q = dict_to_array(pv_preds, horizon, quantiles)
    load_q = dict_to_array(load_preds, horizon, quantiles) / 5e3
    return temp_q, pv_q, load_q


def evaluate_point_forecast_date(date):
    month, day = date
    case_data = build_case_data(month, day, data["solar_data"], data["load_data"], config)
    real_temp, real_net_load = build_data_y_window(month, day, data["solar_data"], data["load_data"])

    real_temp = real_temp.reshape(-1)
    real_net_load = real_net_load.reshape(-1)
    selected_solar = case_data["selected_solar"].reset_index(drop=True)
    selected_load = case_data["selected_load"].reset_index(drop=True)

    true_pv = selected_solar["generation"].iloc[-24 * 4 :].to_numpy()
    true_load = selected_load["load"].iloc[-24 * 4 :].to_numpy() / 5e3

    rows = []
    for t in range(0, len(real_temp), STEP_STRIDE):
        temp_q, pv_q, load_q = forecast_step_quantiles(case_data, t)

        for h in range(1, horizon + 1):
            target_idx = t + h - 1
            if target_idx >= len(real_temp):
                continue

            row = {
                "month": month,
                "day": day,
                "t": t,
                "horizon": h,
                "target_idx": target_idx,
                "temp_true": real_temp[target_idx],
                "temp_pred": temp_q[h - 1, median_idx],
                "pv_true": true_pv[target_idx],
                "pv_pred": pv_q[h - 1, median_idx],
                "load_true": true_load[target_idx],
                "load_pred": load_q[h - 1, median_idx],
                "net_load_true": real_net_load[target_idx],
            }
            row["net_load_pred"] = row["load_pred"] - row["pv_pred"]
            rows.append(row)

    return rows


def point_metrics(df, target_name, true_col, pred_col):
    y_true = df[true_col].to_numpy(dtype=float)
    y_pred = df[pred_col].to_numpy(dtype=float)
    err = y_pred - y_true
    abs_err = np.abs(err)
    mae = np.mean(abs_err)
    rmse = np.sqrt(np.mean(err**2))
    nonzero = np.abs(y_true) > 1e-8
    range_scale = np.max(y_true) - np.min(y_true)
    return pd.Series(
        {
            "target": target_name,
            "n": len(df),
            "MAPE_%": np.mean(abs_err[nonzero] / np.abs(y_true[nonzero])) * 100 if np.any(nonzero) else np.nan,
            "MAE": mae,
            "RMSE": rmse,
            "nMAE_%": mae / range_scale * 100 if range_scale > 1e-8 else np.nan,
            "nRMSE_%": rmse / range_scale * 100 if range_scale > 1e-8 else np.nan,
        }
    )


def point_metrics_table(df):
    return pd.DataFrame(
        [
            point_metrics(df, "temperature", "temp_true", "temp_pred"),
            point_metrics(df, "net_load_scaled", "net_load_true", "net_load_pred"),
        ]
    ).set_index("target")


def point_metrics_by_horizon_table(df):
    rows = []
    for h, group in df.groupby("horizon"):
        current = point_metrics_table(group).reset_index()
        current.insert(0, "horizon", h)
        rows.append(current)
    return pd.concat(rows, ignore_index=True).set_index(["horizon", "target"])


In [5]:
# Point-forecast accuracy is calculated after scenario generation below.
# It uses the generated-scenario mean as the point prediction, so no separate
# median-quantile evaluation pass is needed.


## 4. Quantile and Scenario Diagnostics

The uncertainty scenarios used by MPC are generated from the quantile-based XGBoost forecasts. This section evaluates scenario quality for the two variables actually passed to the MPC optimizers: temperature and net load.

Reported metrics:

- `PICP_90`: empirical coverage of the scenario 5th-95th percentile interval.
- `mean_error`: average scenario mean minus the realized value.
- `scenario_std`: average standard deviation of the generated scenarios.
- `q05`: average 5th percentile of generated scenarios.
- `q95`: average 95th percentile of generated scenarios.


In [6]:
def evaluate_scenario_date(date):
    month, day = date
    case_data = build_case_data(month, day, data["solar_data"], data["load_data"], config)
    real_temp, real_net_load = build_data_y_window(month, day, data["solar_data"], data["load_data"])
    real_temp = real_temp.reshape(-1)
    real_net_load = real_net_load.reshape(-1)

    rows = []
    for t in range(0, len(real_temp), STEP_STRIDE):
        (
            temp_mean,
            net_load_mean,
            temp_scenario_deviation,
            net_load_scenario_deviation,
            _,
        ) = scenario_generator.generate(case_data, t)

        temp_scenarios = temp_scenario_deviation + temp_mean.reshape(1, -1)
        net_load_scenarios = net_load_scenario_deviation + net_load_mean.reshape(1, -1)

        for h in range(1, horizon + 1):
            target_idx = t + h - 1
            if target_idx >= len(real_temp):
                continue

            specs = [
                ("temperature", real_temp[target_idx], temp_scenarios[:, h - 1]),
                ("net_load_scaled", real_net_load[target_idx], net_load_scenarios[:, h - 1]),
            ]
            for target, true_value, scenario_values in specs:
                q05 = np.quantile(scenario_values, 0.05)
                q95 = np.quantile(scenario_values, 0.95)
                scenario_mean = np.mean(scenario_values)
                rows.append(
                    {
                        "month": month,
                        "day": day,
                        "t": t,
                        "horizon": h,
                        "target_idx": target_idx,
                        "target": target,
                        "true": true_value,
                        "scenario_mean": scenario_mean,
                        "mean_error": scenario_mean - true_value,
                        "scenario_std": np.std(scenario_values, ddof=1),
                        "q05": q05,
                        "q95": q95,
                        "covered_90": q05 <= true_value <= q95,
                    }
                )

    return rows


def scenario_metrics_table(df):
    return (
        df.groupby("target")
        .agg(
            n=("true", "size"),
            PICP_90=("covered_90", "mean"),
            mean_error=("mean_error", "mean"),
            scenario_std=("scenario_std", "mean"),
            q05=("q05", "mean"),
            q95=("q95", "mean"),
        )
        .sort_index()
    )


def scenario_metrics_by_horizon_table(df):
    return (
        df.groupby(["horizon", "target"])
        .agg(
            n=("true", "size"),
            PICP_90=("covered_90", "mean"),
            mean_error=("mean_error", "mean"),
            scenario_std=("scenario_std", "mean"),
            q05=("q05", "mean"),
            q95=("q95", "mean"),
        )
        .sort_index()
    )


In [7]:
all_dates = [
    (month, day)
    for month in range(7, 8)
    for day in range(2, 32)
]

EVAL_DATES = all_dates[:]

MAX_DATES = 50 # Change to None for final full holdout results.
STEP_STRIDE = 1

if MAX_DATES is not None:
    EVAL_DATES = EVAL_DATES[-MAX_DATES:]

In [8]:
scenario_rows = []
failed_scenario_dates = []

for date in tqdm(EVAL_DATES, desc="Scenario evaluation"):
    try:
        scenario_rows.extend(evaluate_scenario_date(date))
    except Exception as exc:
        failed_scenario_dates.append((date, repr(exc)))

scenario_eval = pd.DataFrame(scenario_rows)
print(f"Scenario rows: {len(scenario_eval)}")
if failed_scenario_dates:
    print("Failed scenario dates:")
    for item in failed_scenario_dates:
        print(item)

scenario_metrics_overall = scenario_metrics_table(scenario_eval)
scenario_metrics_by_horizon = scenario_metrics_by_horizon_table(scenario_eval)

display(scenario_metrics_overall.round(4))
display(scenario_metrics_by_horizon.round(4))

# Use the mean of the generated scenarios as the point prediction.
point_keys = ["month", "day", "t", "horizon", "target_idx"]
temp_point_eval = (
    scenario_eval.loc[scenario_eval["target"] == "temperature", point_keys + ["true", "scenario_mean"]]
    .rename(columns={"true": "temp_true", "scenario_mean": "temp_pred"})
)
net_load_point_eval = (
    scenario_eval.loc[scenario_eval["target"] == "net_load_scaled", point_keys + ["true", "scenario_mean"]]
    .rename(columns={"true": "net_load_true", "scenario_mean": "net_load_pred"})
)
point_eval = temp_point_eval.merge(net_load_point_eval, on=point_keys, validate="one_to_one")

point_metrics_overall = point_metrics_table(point_eval)
point_metrics_by_horizon = point_metrics_by_horizon_table(point_eval)

print(f"Scenario-mean point forecast rows: {len(point_eval)}")
display(point_metrics_overall.round(4))
display(point_metrics_by_horizon.round(4))


Scenario evaluation:   0%|          | 0/30 [00:00<?, ?it/s]

Scenario evaluation: 100%|██████████| 30/30 [32:53<00:00, 65.79s/it]

Scenario rows: 44400


,n,PICP_90,mean_error,scenario_std,q05,q95
target,,,,,,
net_load_scaled,22200,0.8878,0.6769,3.6435,0.5230,9.6002
temperature,22200,0.7268,-0.0855,0.5640,28.2761,30.0191


n  PICP_90  mean_error  scenario_std      q05  \
horizon target                                                              
1       net_load_scaled  2880   0.8917      1.3049        3.3552   1.3939   
        temperature      2880   0.7104     -0.0372        0.2873  28.7485   
2       net_load_scaled  2850   0.9053      0.9379        3.7026   0.8060   
        temperature      2850   0.7091     -0.0139        0.3865  28.6415   
3       net_load_scaled  2820   0.8897      0.8634        3.7341   0.6779   
        temperature      2820   0.7546     -0.2076        0.5392  28.3300   
4       net_load_scaled  2790   0.8953      0.8563        3.5184   0.7849   
        temperature      2790   0.7638     -0.1511        0.6074  28.2066   
5       net_load_scaled  2760   0.9025      0.8365        3.6640   0.3361   
        temperature      2760   0.7627     -0.1788        0.7216  28.0400   
6       net_load_scaled  2730   0.8736      0.2260        3.6355   0.3360   
        temperature      2730   0.7399      0.0492        0.6653  28.0943   
7       net_load_scaled  2700   0.8685     -0.2310        3.7614  -0.2672   
        temperature      2700   0.7241     -0.0667        0.6922  28.0476   
8       net_load_scaled  2670   0.8742      0.5509        3.7941   0.0277   
        temperature      2670   0.6479     -0.0770        0.6362  28.0534   

                             q95  
horizon target                    
1       net_load_scaled   9.3337  
        temperature      29.6050  
2       net_load_scaled   9.6090  
        temperature      29.8419  
3       net_load_scaled   9.6115  
        temperature      29.9780  
4       net_load_scaled   9.7244  
        temperature      30.0614  
5       net_load_scaled   9.7921  
        temperature      30.1717  
6       net_load_scaled   9.6335  
        temperature      30.2203  
7       net_load_scaled   9.5751  
        temperature      30.2735  
8       net_load_scaled   9.5294  
        temperature      30.0338

Scenario-mean point forecast rows: 22200


,n,MAPE_%,MAE,RMSE,nMAE_%,nRMSE_%
target,,,,,,
temperature,22200,1.9341,0.5902,0.936,4.0272,6.3868
net_load_scaled,22200,130.1043,2.5891,3.099,11.2181,13.4273


n    MAPE_%     MAE    RMSE   nMAE_%  nRMSE_%
horizon target                                                           
1       temperature      2880    1.0206  0.3118  0.4880   2.1274   3.3298
        net_load_scaled  2880  106.0318  2.2037  2.6999   9.5483  11.6981
2       temperature      2850    1.4101  0.4281  0.6669   2.9211   4.5506
        net_load_scaled  2850  139.2784  2.7205  3.1587  11.7875  13.6858
3       temperature      2820    1.8160  0.5608  0.9065   3.8267   6.1852
        net_load_scaled  2820  142.3705  2.7700  3.2378  12.0019  14.0284
4       temperature      2790    1.9421  0.5957  0.9435   4.0646   6.4377
        net_load_scaled  2790  118.0706  2.3149  2.7348  10.0297  11.8493
5       temperature      2760    2.1598  0.6637  1.0419   4.5282   7.1088
        net_load_scaled  2760  127.8734  2.5768  3.0534  11.1647  13.2296
6       temperature      2730    2.2700  0.6871  1.0372   4.6878   7.0769
        net_load_scaled  2730  128.6354  2.4574  2.9313  10.6475  12.7005
7       temperature      2700    2.4058  0.7314  1.0978   4.9904   7.4905
        net_load_scaled  2700  127.7462  2.6270  3.1675  11.3822  13.7241
8       temperature      2670    2.5410  0.7713  1.1466   5.2627   7.8232
        net_load_scaled  2670  152.0898  3.0693  3.7261  13.2985  16.1442

## 5. Export Results


In [9]:
paths.results_dir.mkdir(parents=True, exist_ok=True)

point_metrics_overall.to_csv(paths.results_dir / "reviewer_point_metrics_overall.csv")
point_metrics_by_horizon.to_csv(paths.results_dir / "reviewer_point_metrics_by_horizon.csv")
point_eval.to_csv(paths.results_dir / "reviewer_point_forecast_raw.csv", index=False)

scenario_metrics_overall.to_csv(paths.results_dir / "reviewer_scenario_metrics_overall30.csv")
scenario_metrics_by_horizon.to_csv(paths.results_dir / "reviewer_scenario_metrics_by_horizon30.csv")
scenario_eval.to_csv(paths.results_dir / "reviewer_scenario_raw30.csv", index=False)

print("Saved reviewer diagnostics to:")
print(paths.results_dir)


Saved reviewer diagnostics to:
/Users/liqi/Desktop/UATMPC codes/UATMPC/results


## 6. Reviewer Response Summary Text


In [10]:
print("Point forecasting accuracy (generated-scenario mean as point prediction):")
for target, row in point_metrics_overall.iterrows():
    print(
        f"{target}: MAPE = {row['MAPE_%']:.2f}%, MAE = {row['MAE']:.4f}, "
        f"RMSE = {row['RMSE']:.4f}, nMAE = {row['nMAE_%']:.2f}%, "
        f"nRMSE = {row['nRMSE_%']:.2f}% (n = {int(row['n'])})."
    )

print()
print("Scenario diagnostics:")
for target, row in scenario_metrics_overall.iterrows():
    print(
        f"{target}: PICP = {row['PICP_90']:.3f}, mean error = {row['mean_error']:.4f}, "
        f"scenario std = {row['scenario_std']:.4f}, q05 = {row['q05']:.4f}, "
        f"q95 = {row['q95']:.4f}."
    )


Point forecasting accuracy (generated-scenario mean as point prediction):
temperature: MAPE = 1.93%, MAE = 0.5902, RMSE = 0.9360, nMAE = 4.03%, nRMSE = 6.39% (n = 22200).
net_load_scaled: MAPE = 130.10%, MAE = 2.5891, RMSE = 3.0990, nMAE = 11.22%, nRMSE = 13.43% (n = 22200).

Scenario diagnostics:
net_load_scaled: PICP = 0.888, mean error = 0.6769, scenario std = 3.6435, q05 = 0.5230, q95 = 9.6002.
temperature: PICP = 0.727, mean error = -0.0855, scenario std = 0.5640, q05 = 28.2761, q95 = 30.0191.


In [11]:
print("Point forecasting accuracy (generated-scenario mean as point prediction):")
for target, row in point_metrics_overall.iterrows():
    print(
        f"{target}: MAPE = {row['MAPE_%']:.2f}%, MAE = {row['MAE']:.4f}, "
        f"RMSE = {row['RMSE']:.4f}, nMAE = {row['nMAE_%']:.2f}%, "
        f"nRMSE = {row['nRMSE_%']:.2f}% (n = {int(row['n'])})."
    )

print()
print("Scenario diagnostics:")
for target, row in scenario_metrics_overall.iterrows():
    print(
        f"{target}: PICP = {row['PICP_90']:.3f}, mean error = {row['mean_error']:.4f}, "
        f"scenario std = {row['scenario_std']:.4f}, q05 = {row['q05']:.4f}, "
        f"q95 = {row['q95']:.4f}."
    )


Point forecasting accuracy (generated-scenario mean as point prediction):
temperature: MAPE = 1.93%, MAE = 0.5902, RMSE = 0.9360, nMAE = 4.03%, nRMSE = 6.39% (n = 22200).
net_load_scaled: MAPE = 130.10%, MAE = 2.5891, RMSE = 3.0990, nMAE = 11.22%, nRMSE = 13.43% (n = 22200).

Scenario diagnostics:
net_load_scaled: PICP = 0.888, mean error = 0.6769, scenario std = 3.6435, q05 = 0.5230, q95 = 9.6002.
temperature: PICP = 0.727, mean error = -0.0855, scenario std = 0.5640, q05 = 28.2761, q95 = 30.0191.
